# Краткое описание
- Цель эксперимента - обучить модель для разметки токсичных спанов.
- Данные - ru_toxic_spans после подготовки и разбиения.
- Основные выводы - выбрана лучшая модель по метрикам и сохранены конфиги.


In [40]:

from pathlib import Path
import json
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


import os
import re
import json
import logging
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional
from collections import Counter
from functools import partial
import random


import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns


from tqdm.auto import tqdm


import joblib
import pickle


from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, average_precision_score, classification_report,
    precision_recall_curve, confusion_matrix
)


from scipy import sparse


try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit
except Exception:
    MultilabelStratifiedKFold = None
    MultilabelStratifiedShuffleSplit = None


try:

    from gensim.models import KeyedVectors
except Exception:
    KeyedVectors = None

try:
    import fasttext
    import fasttext.util
except Exception:
    fasttext = None


try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from torch.optim import AdamW
except Exception:
    torch = None


try:
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,
        TrainingArguments, Trainer, DataCollatorWithPadding
    )
except Exception:
    AutoTokenizer = AutoModelForSequenceClassification = TrainingArguments = Trainer = DataCollatorWithPadding = None


import warnings
warnings.filterwarnings("ignore")


sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

In [41]:
def set_seed(seed: int = 42) -> None:
    # Фиксируем seed для воспроизводимости (насколько это возможно).
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass


    global DATA_LOADER_GEN
    try:
        DATA_LOADER_GEN = torch.Generator()
        DATA_LOADER_GEN.manual_seed(seed)
    except Exception:
        DATA_LOADER_GEN = None

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [42]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("textdetox/multilingual_toxic_spans", split="ru")

In [43]:
df = pd.DataFrame(ds)

In [44]:
df.head()

,Sentence,Negative Connotations
0,во дебил ты пан-баран))))))))))))))))))). евре...,"дебил, пан-баран, еврейское тв, даун"
1,завтра в хабаровске вдвшники будут разгонять б...,быдло
2,"эта походу старое фото,в середине хряк телепуз...","хряк, телепузик, редкая скотина"
3,а что рядом других что мужиков рядом не оказал...,"ублюдку, табло, перевелись"
4,"хули вы бляди клянчите классы ,просто помните.","хули,бляди"


In [45]:
df.shape

(999, 2)

In [46]:
def tokenize(text):
    return re.findall(r'\w+|[^\w\s]', text.lower(), re.UNICODE)

def buildlabels(tokens, toxicwords):
    labels = [0] * len(tokens)

    toxiclist = [
        tokenize(word.strip())
        for word in toxicwords.split(',')
        if word.strip()
    ]

    for toxic in toxiclist:
        n = len(toxic)

        for i in range(len(tokens) - n + 1):
            if tokens[i:i+n] == toxic:
                for j in range(i, i+n):
                    labels[j] = 1

    return labels

df["tokens"] = df["Sentence"].apply(tokenize)

df["labels"] = df.apply(
    lambda row: buildlabels(
        row["tokens"],
        row["Negative Connotations"]
    ),
    axis=1
)

df[["tokens", "labels"]].head()

,tokens,labels
0,"[во, дебил, ты, пан, -, баран, ), ), ), ), ), ...","[0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,"[завтра, в, хабаровске, вдвшники, будут, разго...","[0, 0, 0, 0, 0, 0, 1]"
2,"[эта, походу, старое, фото, ,, в, середине, хр...","[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0]"
3,"[а, что, рядом, других, что, мужиков, рядом, н...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
4,"[хули, вы, бляди, клянчите, классы, ,, просто,...","[1, 0, 1, 0, 0, 0, 0, 0, 0]"


In [47]:
metrics_df = pd.DataFrame()

In [48]:

from torch.utils.data import TensorDataset

class RNNTagger(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 100, hidden_size: int = 128,
                 num_layers: int = 1, dropout: float = 0.2, rnn_type: str = 'GRU'):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        rnn_type = rnn_type.upper()
        if rnn_type == 'LSTM':
            self.rnn = nn.LSTM(embed_dim, hidden_size, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers>1 else 0.0)
        elif rnn_type == 'RNN':
            self.rnn = nn.RNN(embed_dim, hidden_size, num_layers=num_layers, batch_first=True, nonlinearity='tanh', dropout=dropout if num_layers>1 else 0.0)
        else:
            self.rnn = nn.GRU(embed_dim, hidden_size, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers>1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.rnn(emb)
        out = self.dropout(out)
        logits = self.head(out).squeeze(-1)
        return logits

def build_vocab_from_tokens(token_lists, min_freq=1):
    freq = Counter()
    for toks in token_lists:
        for t in toks:
            freq[t] += 1
    word2idx = {'<PAD>': 0, '<UNK>': 1}
    idx = 2
    for w, c in freq.items():
        if c >= min_freq:
            word2idx[w] = idx
            idx += 1
    return word2idx

def tokens_to_padded_indices(tokens_list, w2i, max_len):
    seqs = []
    masks = []
    for toks in tokens_list:
        idxs = [w2i.get(t, w2i.get('<UNK>')) for t in toks][:max_len]
        mask = [1]*len(idxs)
        if len(idxs) < max_len:
            pad = [w2i['<PAD>']] * (max_len - len(idxs))
            idxs = idxs + pad
            mask = mask + [0]*(max_len - len(mask))
        seqs.append(idxs)
        masks.append(mask)
    return np.array(seqs, dtype=np.int64), np.array(masks, dtype=np.float32)

def labels_to_padded(labels_list, max_len):
    arr = []
    for labs in labels_list:
        labs = labs[:max_len]
        if len(labs) < max_len:
            labs = labs + [0]*(max_len - len(labs))
        arr.append(labs)
    return np.array(arr, dtype=np.float32)


if 'train_df' not in globals() or 'val_df' not in globals() or 'test_df' not in globals():
    df = df.copy()
    df['has_toxic'] = df['labels'].apply(lambda lbl: int(any(lbl)))
    train_df, temp_df = train_test_split(df, test_size=0.40, random_state=42, stratify=df['has_toxic'])
    temp_df = temp_df.copy()
    temp_df['has_toxic'] = temp_df['labels'].apply(lambda lbl: int(any(lbl)))
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['has_toxic'])

max_len = 100
word2idx = build_vocab_from_tokens(train_df['tokens'].tolist(), min_freq=1)

X_train_seq, X_train_mask = tokens_to_padded_indices(train_df['tokens'].tolist(), word2idx, max_len)
X_val_seq, X_val_mask = tokens_to_padded_indices(val_df['tokens'].tolist(), word2idx, max_len)
X_test_seq, X_test_mask = tokens_to_padded_indices(test_df['tokens'].tolist(), word2idx, max_len)
y_train = labels_to_padded(train_df['labels'].tolist(), max_len)
y_val = labels_to_padded(val_df['labels'].tolist(), max_len)
y_test = labels_to_padded(test_df['labels'].tolist(), max_len)

train_dataset = TensorDataset(torch.LongTensor(X_train_seq), torch.FloatTensor(y_train), torch.FloatTensor(X_train_mask))
val_dataset = TensorDataset(torch.LongTensor(X_val_seq), torch.FloatTensor(y_val), torch.FloatTensor(X_val_mask))
test_dataset = TensorDataset(torch.LongTensor(X_test_seq), torch.FloatTensor(y_test), torch.FloatTensor(X_test_mask))

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

def train_and_evaluate(model, name, epochs=5, lr=1e-3):
    model = model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss(reduction='none')
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_tokens = 0
        for Xb, yb, mb in train_loader:
            Xb = Xb.to(device)
            yb = yb.to(device)
            mb = mb.to(device)
            optimizer.zero_grad()
            logits = model(Xb)
            loss = loss_fn(logits, yb)
            loss = (loss * mb).sum() / (mb.sum() + 1e-9)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * mb.sum().item()
            total_tokens += mb.sum().item()
        avg_loss = total_loss / (total_tokens + 1e-9)
        # validation
        model.eval()
        y_true_all = []
        y_score_all = []
        with torch.no_grad():
            for Xb, yb, mb in val_loader:
                Xb = Xb.to(device)
                yb = yb.to(device)
                mb = mb.to(device)
                logits = model(Xb)
                probs = torch.sigmoid(logits)
                y_true_all.extend(yb[mb==1].cpu().numpy().tolist())
                y_score_all.extend(probs[mb==1].cpu().numpy().tolist())
        if len(set(y_true_all)) == 1:
            roc = None
        else:
            try:
                roc = float(roc_auc_score(y_true_all, y_score_all))
            except Exception:
                roc = None
        preds = [1 if s>=0.5 else 0 for s in y_score_all]
        acc = accuracy_score(y_true_all, preds) if len(y_true_all)>0 else 0.0
        prec = precision_score(y_true_all, preds, zero_division=0) if len(y_true_all)>0 else 0.0
        rec = recall_score(y_true_all, preds, zero_division=0) if len(y_true_all)>0 else 0.0
        print(f"{name} epoch {epoch+1}/{epochs} loss={avg_loss:.6f} val_acc={acc:.4f} prec={prec:.4f} rec={rec:.4f} roc_auc={roc}")
    # final metrics
    return {'model': name, 'accuracy': float(acc), 'precision': float(prec), 'recall': float(rec), 'roc_auc': roc}

metrics_list = []
models_dict = {}

# Train RNN, GRU, LSTM variants and keep trained models
for rnn_type in ['RNN', 'GRU', 'LSTM']:
    name = f"{rnn_type}-tagger"
    model = RNNTagger(vocab_size=len(word2idx), embed_dim=100, hidden_size=128, num_layers=2, dropout=0.3, rnn_type=rnn_type)
    metrics_res = train_and_evaluate(model, name=name, epochs=5, lr=1e-3)
    metrics_list.append(metrics_res)
    models_dict[name] = model

metrics_df_trie = pd.DataFrame(metrics_list)
if 'metrics_df' in globals():
    metrics_df = pd.concat([metrics_df, metrics_df_trie], ignore_index=True)
else:
    metrics_df = metrics_df_trie

display(metrics_df)

# Выбор лучшей модели по roc_auc и сохранение конфигов + весов
metrics_df['roc_auc'] = pd.to_numeric(metrics_df.get('roc_auc'), errors='coerce')
metrics_sorted = metrics_df.sort_values(by='roc_auc', ascending=False, na_position='last').reset_index(drop=True)
print('\nSorted metrics by roc_auc:')
display(metrics_sorted)

best_row = metrics_sorted.iloc[0]
best_name = best_row['model']
print('\nBest by roc_auc:', best_name, best_row.to_dict())

config_dir = Path("../../configs/spans_model").resolve()
config_dir.mkdir(parents=True, exist_ok=True)

hyperparams = {"model": best_name}
train_params = {}
inference_params = {}

try:
    model_obj = models_dict.get(best_name) if 'models_dict' in globals() else None
    if model_obj is not None:
        if 'word2idx' in globals():
            hyperparams['vocab_size'] = int(len(word2idx))
        emb = getattr(model_obj, 'embedding', None)
        if emb is not None:
            hyperparams['embed_dim'] = int(getattr(emb, 'embedding_dim', None))
        head = getattr(model_obj, 'head', None)
        if head is not None:
            hyperparams['hidden_size'] = int(getattr(head, 'in_features', None))
        rnn_module = getattr(model_obj, 'rnn', None)
        if rnn_module is not None:
            hyperparams['num_layers'] = int(getattr(rnn_module, 'num_layers', None) or 0)
        dropout_obj = getattr(model_obj, 'dropout', None)
        if dropout_obj is not None:
            hyperparams['dropout'] = float(getattr(dropout_obj, 'p', None))
        # rnn type parsed from name
        hyperparams['rnn_type'] = str(best_name).split('-')[0]

    train_params = {"epochs": 5, "lr": 1e-3, "batch_size": int(batch_size), "optimizer": "AdamW"}
    inference_params = {"max_len": int(max_len), "threshold": 0.5}

    # save jsons
    with open(config_dir / 'hyperparams.json', 'w', encoding='utf-8') as f:
        json.dump(hyperparams, f, ensure_ascii=False, indent=2)
    with open(config_dir / 'train_params.json', 'w', encoding='utf-8') as f:
        json.dump(train_params, f, ensure_ascii=False, indent=2)
    with open(config_dir / 'inference_params.json', 'w', encoding='utf-8') as f:
        json.dump(inference_params, f, ensure_ascii=False, indent=2)

    print('Saved configs to', config_dir)

    # save model weights and word2idx
    artifacts_dir = Path("../../artifacts/spans_model").resolve()
    artifacts_dir.mkdir(parents=True, exist_ok=True)

    with open(artifacts_dir / "word2idx.json", "w", encoding="utf-8") as f:
        json.dump(word2idx, f, ensure_ascii=False, indent=2)

    saved = False
    if model_obj is not None and hasattr(model_obj, 'state_dict') and torch is not None:
        torch.save(model_obj.state_dict(), str(artifacts_dir / 'best_spans_model.pt'))
        print('Saved torch state_dict to', artifacts_dir / 'best_spans_model.pt')
        saved = True
    else:
        # fallback: try to save any sklearn estimator via joblib if present
        est = None
        for cand in ['best_estimator_', 'dt_grid', 'gb_grid', 'svm_grid', 'dummy']:
            if cand in globals():
                obj = globals().get(cand)
                if hasattr(obj, 'fit') or hasattr(obj, 'predict'):
                    est = obj
                    break
        if est is not None:
            try:
                joblib.dump(est, str(artifacts_dir / 'best_spans_model.joblib'))
                print('Saved sklearn estimator to', artifacts_dir / 'best_spans_model.joblib')
                saved = True
            except Exception:
                pass

    if not saved:
        print('No model artifact saved: trained model not found to save state_dict or sklearn estimator.')
    # Сохраняем метрики в config_dir/metrics.json
    try:
        metrics_out = metrics_sorted.where(pd.notnull(metrics_sorted), None).to_dict(orient='records')
    except Exception:
        try:
            metrics_out = metrics_df.where(pd.notnull(metrics_df), None).to_dict(orient='records')
        except Exception:
            metrics_out = []

    with open(config_dir / 'metrics.json', 'w', encoding='utf-8') as f:
        json.dump(metrics_out, f, ensure_ascii=False, indent=2)

    print('Saved metrics to', config_dir / 'metrics.json')
except Exception as e:
    print('Error while saving configs or model:', e)


RNN-tagger epoch 1/5 loss=0.494866 val_acc=0.8626 prec=0.5625 rec=0.0255 roc_auc=0.7108583144668983
RNN-tagger epoch 2/5 loss=0.380191 val_acc=0.8645 prec=0.6296 rec=0.0482 roc_auc=0.7299341144187614
RNN-tagger epoch 3/5 loss=0.353869 val_acc=0.8720 prec=0.7600 rec=0.1076 roc_auc=0.7442721760518333
RNN-tagger epoch 4/5 loss=0.331450 val_acc=0.8790 prec=0.7750 rec=0.1756 roc_auc=0.7596604942641318
RNN-tagger epoch 5/5 loss=0.308633 val_acc=0.8798 prec=0.7500 rec=0.1955 roc_auc=0.7517629766536715
GRU-tagger epoch 1/5 loss=0.509263 val_acc=0.8571 prec=0.2273 rec=0.0142 roc_auc=0.6068906355982923
GRU-tagger epoch 2/5 loss=0.403343 val_acc=0.8610 prec=0.4000 rec=0.0113 roc_auc=0.6827793959222759
GRU-tagger epoch 3/5 loss=0.372887 val_acc=0.8618 prec=0.5000 rec=0.0255 roc_auc=0.766474291237694
GRU-tagger epoch 4/5 loss=0.339189 val_acc=0.8661 prec=0.6486 rec=0.0680 roc_auc=0.812038179915645
GRU-tagger epoch 5/5 loss=0.312505 val_acc=0.8712 prec=0.7000 rec=0.1190 roc_auc=0.8307922100822057
LS

,model,accuracy,precision,recall,roc_auc
0,RNN-tagger,0.879796,0.75,0.195467,0.751763
1,GRU-tagger,0.871182,0.70,0.118980,0.830792
2,LSTM-tagger,0.865701,1.00,0.028329,0.752715



Sorted metrics by roc_auc:


,model,accuracy,precision,recall,roc_auc
0,GRU-tagger,0.871182,0.70,0.118980,0.830792
1,LSTM-tagger,0.865701,1.00,0.028329,0.752715
2,RNN-tagger,0.879796,0.75,0.195467,0.751763



Best by roc_auc: GRU-tagger {'model': 'GRU-tagger', 'accuracy': 0.8711824588880188, 'precision': 0.7, 'recall': 0.11898016997167139, 'roc_auc': 0.8307922100822057}
Saved configs to D:\konst\PycharmProjects\mirea_real\AIE_DPO\project\configs\spans_model
Saved torch state_dict to D:\konst\PycharmProjects\mirea_real\AIE_DPO\project\artifacts\spans_model\best_spans_model.pt
Saved metrics to D:\konst\PycharmProjects\mirea_real\AIE_DPO\project\configs\spans_model\metrics.json


In [49]:
model_eval = None
if 'model_obj' in globals() and model_obj is not None:
    model_eval = model_obj
elif 'best_name' in globals() and 'models_dict' in globals():
    model_eval = models_dict.get(best_name)
else:
    model_eval = None

if model_eval is None:
    raise ValueError("model for evaluation not found")

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
model_eval = model_eval.to(device)
model_eval.eval()

y_true_all = []
y_score_all = []
with torch.no_grad():
    for Xb, yb, mb in test_loader:
        Xb = Xb.to(device)
        yb = yb.to(device)
        mb = mb.to(device)
        logits = model_eval(Xb)
        probs = torch.sigmoid(logits)
        y_true_all.extend(yb[mb == 1].cpu().numpy().tolist())
        y_score_all.extend(probs[mb == 1].cpu().numpy().tolist())

if len(set(y_true_all)) == 1:
    roc = None
else:
    try:
        roc = float(roc_auc_score(y_true_all, y_score_all))
    except Exception:
        roc = None

preds = [1 if s >= 0.5 else 0 for s in y_score_all]
metrics = {
    "accuracy": float(accuracy_score(y_true_all, preds)) if len(y_true_all) > 0 else 0.0,
    "precision": float(precision_score(y_true_all, preds, zero_division=0)) if len(y_true_all) > 0 else 0.0,
    "recall": float(recall_score(y_true_all, preds, zero_division=0)) if len(y_true_all) > 0 else 0.0,
    "f1": float(f1_score(y_true_all, preds, zero_division=0)) if len(y_true_all) > 0 else 0.0,
    "roc_auc": roc,
}

with open(artifacts_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print("Saved metrics to", artifacts_dir / "metrics.json")


Saved metrics to D:\konst\PycharmProjects\mirea_real\AIE_DPO\project\artifacts\spans_model\metrics.json
